- why async & how
    - Agent 任务的“长尾效应” (The Straggler Problem)
        - 样本 A: 简单问题 -> 1 轮对话 -> 结束。
        - 样本 B: 复杂问题 -> 思考 -> 调用工具 -> 报错 -> 修正 -> 再调用 -> 5 轮对话 -> 结束。
        - 如果在 Trainer 中使用同步 Batch 生成，整个 Batch 的生成时间取决于最慢的那一个样本（Straggler）。GPU 会在大部分样本生成完后空转，等待那个“长尾”样本，导致极大的算力浪费。
    - 提升 CPU 密集型任务的并发能力 (CPU-Intensive Tasks)
        - Agent 任务通常包含大量的 CPU 密集型操作，例如：
            - 工具执行：运行 Python 代码解释器、搜索网页。
            - 环境模拟：运行编译器、单元测试。
            - 奖励计算：运行复杂的规则检查。
        - 在 AgentLoopManager 中，AgentLoopWorker 被调度到 CPU 节点上运行。
            - GPU 专注推理：昂贵的 GPU 资源只用于 LLM 的 generate 调用。
            - CPU 并发交互：大量的工具调用和环境交互在廉价的 CPU 节点上异步并发执行。
            - 如果不使用 Async 模式，GPU 进程可能需要挂起等待 CPU 完成工具调用，导致显存占用且算力闲置。
- agent loop 状态管理
    - multi-turn (config): tool or user(interaction) simulator 
- token in token output：避免 Chat Template 在多轮对话中反复 Encode/Decode 导致的不一致问题
    - `AsyncLLMServerManager:generate`
        - `prompt_ids: list[int],`：输入是 Token IDs，而不是 Chat Messages
- reward 计算与计算时机
    - 在 Async 模式（即 rollout.mode="async"）下，Reward 的计算被前置到了 Rollout 阶段内部完成，
- loss/advantage 计算
- llm server manager 负载均衡

-  时序图 (Sequence Diagram)。
    -  展示对象之间（核心抽象）交互的时间顺序，清晰地表达了“谁在什么时候调用了谁”以及“数据是如何在不同模块间流转的”。
    -  控制流（Control Flow） 和 数据流（Data Flow）

<div style="width: 70%; margin: auto;">
    
```mermaid
sequenceDiagram
    participant Trainer as RayPPOTrainer
    participant Manager as AgentLoopManager
    participant Worker as AgentLoopWorker
    participant RM as RewardManagerWorker

    Trainer->>Manager: generate_sequences()
    Manager->>Worker: distribute tasks
    
    rect rgb(240, 248, 255)
        Note over Worker: Async Rollout Phase
        Worker->>Worker: agent_loop.run() <br/> (Interaction/Generation)
        
        Note over Worker: Reward Calculation Trigger
        Worker->>RM: compute_score(Full Trajectory)
        RM-->>Worker: return reward_score
        
        Worker->>Worker: _postprocess() <br/> (Attach score to last token)
    end
    
    Worker-->>Manager: Return Batch (with rm_scores)
    Manager-->>Trainer: Return Batch
    
    Note over Trainer: Training Phase
    Trainer->>Trainer: Check "rm_scores" exists?
    Trainer->>Trainer: SKIP compute_rm_score()
    Trainer->>Trainer: Compute Advantage & Update
```

- AgentLoopManager 将 Batch 拆分成多个 Chunk 分发给 AgentLoopWorker。
- Worker 内部利用 asyncio 并发处理多个 Request。
- 配合 vLLM/SGLang 的 Continuous Batching 机制，推理引擎可以随时插入新请求，而不需要等待当前 Batch 全部结束。这使得“快任务”和“慢任务”可以流水线式地交织执行，最大化 GPU 利用率。

### response mask

> tool_agent_loop.py

```python
# LLM 生成的 token: mask = 1                                                                                                                                       
agent_data.response_mask += [1] * len(agent_data.response_ids)
# Tool 返回的 token: mask = 0 
agent_data.response_mask += [0] * len(response_ids)
```
- 这样在计算 PPO loss 时（core_algos.py:216-259），只对 LLM 生成的 token 计算梯度，tool 返回的内容不参与 policy gradient。 

### AgentLoopManager & AsyncLLMServerManager

- AgentLoopManager
    - workers: cpu, num_workers (配置文件配置)
    - 管理状态机，执行工具调用，处理 interaction，async 协程调度
- AsyncLLMServerManager：
    - llm inference
        - 负载均衡 + Sticky Session
    - server (replica): gpu
        - tensor_model_parallel_size * data_parallel_size

```python
def _initialize_llm_servers(self):
    # 单个 replica 需要的 GPU 数
    rollout_world_size = (
        tensor_model_parallel_size  # e.g., 4 (模型切分到4张卡)
        * data_parallel_size        # e.g., 1
        * pipeline_model_parallel_size # e.g., 1
    )

    # 总 GPU 数
    world_size = n_gpus_per_node * nnodes  # e.g., 8 * 2 = 16

    # Server 数量 = 总 GPU / 单 replica GPU
    num_replicas = world_size // rollout_world_size  # e.g., 16 / 4 = 4 servers
```

```
示例配置：
# 16 GPU 集群, 模型需要 4 张卡做 TP
trainer:
  n_gpus_per_node: 8
  nnodes: 2

rollout:
  tensor_model_parallel_size: 4
  data_parallel_size: 1
```
-  4 个 LLM Server replicas  

```python
#Step 1: Manager 分发到 Workers (agent_loop.py:766-772)
def generate_sequences(self, prompts: DataProto) -> DataProto:
  # 把 batch 分成多个 chunk，分发给多个 worker
  chunkes = prompts.chunk(len(self.agent_loop_workers))
  outputs = ray.get([              
      worker.generate_sequences.remote(chunk)
      for worker, chunk in zip(self.agent_loop_workers, chunkes)
  ])                    

# Step 2: Worker 内部为每个 sample 创建 async task (agent_loop.py:360-364)
async def generate_sequences(self, batch: DataProto) -> DataProto:
  tasks = []                       
  for i in range(len(batch)):     
      kwargs = {k: v[i] for k, v in batch.non_tensor_batch.items()}
      # 每个 sample 一个独立的 async task
      tasks.append(asyncio.create_task(self._run_agent_loop(sampling_params, trajectory_info[i], **kwargs)))
  outputs = await asyncio.gather(*tasks)  # 并发执行
                                                                                                                                                                                                                                          
# Step 3: 每个 task 运行状态机 (tool_agent_loop.py:119-166)
async def run(self, sampling_params: dict[str, Any], **kwargs) -> AgentLoopOutput:
  request_id = uuid4().hex  # ← Trajectory 级别的 ID
  agent_data = AgentData(request_id=request_id, ...)         

  # 状态机循环                     
  state = AgentState.PENDING            
  while state != AgentState.TERMINATED:
      if state == AgentState.PENDING:
          state = await self._handle_pending_state(agent_data, sampling_params)                                                                                                                                                       
      elif state == AgentState.GENERATING:      
          state = await self._handle_generating_state(agent_data, sampling_params)
          # ↑ 这里调用 server_manager.generate()
      elif state == AgentState.PROCESSING_TOOLS:
          state = await self._handle_processing_tools_state(agent_data)
      elif state == AgentState.INTERACTING:
          state = await self._handle_interacting_state(agent_data)
                               
# Step 4: 状态机调用 LLM Server (tool_agent_loop.py:221-227)
async def _handle_generating_state(self, agent_data, sampling_params):
  output = await self.server_manager.generate(
      request_id=agent_data.request_id,  # ← Trajectory 级别的 ID，用于 sticky session
      prompt_ids=agent_data.prompt_ids,
      sampling_params=sampling_params, 
  )
```

### agent 状态机

```python
state = AgentState.PENDING

while state != AgentState.TERMINATED:
    if state == AgentState.PENDING:
        state = await self._handle_pending_state(agent_data, sampling_params)
    elif state == AgentState.GENERATING:
        state = await self._handle_generating_state(agent_data, sampling_params)
    elif state == AgentState.PROCESSING_TOOLS:
        state = await self._handle_processing_tools_state(agent_data)
    elif state == AgentState.INTERACTING:
        state = await self._handle_interacting_state(agent_data)
    else:
        logger.error(f"Invalid state: {state}")
        state = AgentState.TERMINATED
```

```mermaid
stateDiagram-v2
    [*] --> PENDING: Start
    
    state "PENDING (初始化)" as PENDING
    state "GENERATING (模型生成)" as GENERATING
    state "PROCESSING_TOOLS (执行工具)" as PROCESSING_TOOLS
    state "INTERACTING (环境/用户交互)" as INTERACTING
    state "TERMINATED (结束)" as TERMINATED

    PENDING --> GENERATING : 准备好 Prompt

    GENERATING --> PROCESSING_TOOLS : 模型决定调用工具
    GENERATING --> INTERACTING : 无工具调用 & 配置了交互环境
    GENERATING --> TERMINATED : 完成任务 / 达到最大轮数 / 达到长度限制

    PROCESSING_TOOLS --> GENERATING : 工具执行完毕 (更新上下文)
    PROCESSING_TOOLS --> TERMINATED : 上下文超长

    INTERACTING --> GENERATING : 收到外部反馈 (更新上下文)
    INTERACTING --> TERMINATED : 交互结束 (should_terminate=True)

    TERMINATED --> [*]: End
```

  状态: PENDING                                                                                                                     
  处理函数: _handle_pending_state                                                                                                   
  主要职责: 初始化 prompt_ids，apply_chat_template                                                                                  
  输出: → GENERATING                                                                                                                
  ────────────────────────────────────────                                                                                          
  状态: GENERATING                                                                                                                  
  处理函数: _handle_generating_state                                                                                                
  主要职责: 调用 LLM 生成，解析 tool_calls                                                                                          
  输出: → PROCESSING_TOOLS / INTERACTING / TERMINATED                                                                               
  ────────────────────────────────────────                                                                                          
  状态: PROCESSING_TOOLS                                                                                                            
  处理函数: _handle_processing_tools_state                                                                                          
  主要职责: 并行执行工具，收集结果                                                                                                  
  输出: → GENERATING / TERMINATED                                                                                                   
  ────────────────────────────────────────                                                                                          
  状态: INTERACTING                                                                                                                 
  处理函数: _handle_interacting_state                                                                                               
  主要职责: 调用环境获取反馈                                                                                                        
  输出: → GENERATING / TERMINATED  

```python
# AgentData 在整个状态机中维护 (tool_agent_loop.py:45-80)                                                                         
class AgentData:
    # 累积的 trajectory                                                                                                           
    prompt_ids: list[int] = []      # 不断追加                                                                                    
    response_mask: list[int] = []   # 标记哪些是 LLM 生成的                                                                       
    response_logprobs: list[float] = []                                                                                           
                                                                                                                                
    # 中间奖励                                                                                                                    
    turn_scores: list[float] = []   # 每轮 interaction 的 reward                                                                  
    tool_rewards: list[float] = []  # 每次工具调用的 reward                                                                       
                                                                                                                                
    # 计数器                                                                                                                      
    user_turns = 0                                                                                                                
    assistant_turns = 0
```

### agent loop 的异步性

```
360|        tasks = []
361|        for i in range(len(batch)):
            # ... 准备参数 ...
363|            tasks.append(asyncio.create_task(self._run_agent_loop(sampling_params, ...)))
364|        outputs = await asyncio.gather(*tasks)
```

- 同步模式 (Sync)：通常是把整个 Batch 作为一个 Tensor 扔给模型，模型一步步生成。所有样本必须“齐步走”。
- 异步模式 (Async)：这里使用了 asyncio.create_task。假设 Batch Size 是 64，代码会瞬间启动 64 个独立的协程。
    - 协程 A 可能正在等待 vLLM 生成 Token。
    - 协程 B 可能正在等待 Tool 返回结果（IO 操作）。
    - 协程 C 可能正在解析 XML 格式。
    - 它们互不阻塞。

### multiturn feedback 以及任务的终止

```python
 async def generate_response(self, instance_id, messages, **kwargs):                                                               
      reward = await self.calculate_score(instance_id)                                                                              
      if reward == 1.0:  # 答对了                                                                                                   
          response = "Your response is correct!"                                                                                    
          should_terminate_sequence = True  # 终止                                                                                  
      else:                                                                                                                         
          response = "Your response is incorrect! You need to reflect on your answer and try again."                                                              
          should_terminate_sequence = False  # 继续                                                                                 
      return should_terminate_sequence, response, reward, {}      
```
- tool & interaction
    - tool
    - interaction: 环境/用户模拟器

### async rollout

  Sample A: Prompt → LLM → Answer (2轮，很快完成)                                                                                   
  Sample B: Prompt → LLM → Tool → LLM → Tool → LLM → Answer (6轮，很慢)                                                             
  Sample C: Prompt → LLM → Interaction → LLM → ... (需要外部 API)                                                                   
                                                                                                                                    
  如果用同步模式：                                                                                                                  
  - Sample A 完成后要等 Sample B、C                                                                                                 
  - GPU 空闲，利用率低                                                                                                              
  - 无法利用 continuous batching     

- Batch 内并发：不同 sample 的 trajectory 长度不同（`agent_loop.py`），让不同长度的 trajectory 能并发执行     

```python
async def generate_sequences(self, batch: DataProto) -> DataProto:
    # 1. 为每个 sample 创建异步任务
    tasks = []                  
    for i in range(len(batch)):  
        kwargs = {k: v[i] for k, v in batch.non_tensor_batch.items()}
        tasks.append(asyncio.create_task(self._run_agent_loop(...)))
                        
    # 2. 等待所有 sample 完成 ← 同步点
    outputs = await asyncio.gather(*tasks)
                
    # 3. 合并结果成 batch                                                                                                                
    output = self._postprocess(outputs)  
    return output
```
    
- 工具调用可以并行：`tool_agent_loop.py`
    - 让多个工具调用能并行 
```python                                              
tasks = []                                                                                                                      for tool_call in agent_data.tool_calls[:self.max_parallel_calls]: 
    tasks.append(self._call_tool(tool_call, agent_data.tools_kwargs))                                                           responses = await asyncio.gather(*tasks)  # 多个工具并发执行   
```

- Sync 模式下“每轮都要等待”，Sync 模式把“一批人”绑在了一起，而 Async 模式把“每个人”拆开了。
- 同步模式 (Sync):强制绑定的“旅行团”
    - 因为在代码层面，llm.generate(batch) 是一个同步阻塞调用。
    - 木桶效应：只要 Batch 里有一个 Agent 没生成完（比如它生成了 500 个 token，而别人只生成了 10 个），整个 generate 函数就不会返回。
    - 工具执行的同步：更重要的是，LLM 生成和工具执行是交替的。Agent 2 即使生成快，它想进行下一轮 LLM 生成，必须先经过“工具执行”阶段。但在 Sync 模式下，代码必须等 Agent 1 也执行完工具，拼好一个新的 Batch，才能统一发起第二次 llm.generate。
    - 结果：Agent 2 在 Round 1 结束后，明明可以立刻开始 Round 2，但它被“绑”在 Batch 里，必须在原地等 Agent 1 慢吞吞地跑完。
```python
# Sync 模式伪代码
prompts = [Agent1_Input, Agent2_Input] # 一个 Batch

# Round 1
# 【卡点 A】：必须等 Agent 1 和 Agent 2 都生成完，函数才返回
outputs_round_1 = llm.generate(prompts) 

# 【卡点 B】：必须等 Agent 1 和 Agent 2 都执行完工具，才能继续
# 假设 Agent 1 要下载大文件（慢），Agent 2 只是 print（快）
# Agent 2 执行完了，但代码卡在这里等 Agent 1
observations = run_tools(outputs_round_1) 

# 构造 Round 2 的输入
prompts_round_2 = update_history(prompts, outputs_round_1, observations)

# Round 2
# 【卡点 C】：再次发起生成，Agent 2 必须等 Agent 1 搞定 Round 1 才能进入这里
outputs_round_2 = llm.generate(prompts_round_2)
```

- 异步模式 (Async)：各跑各的“自由行”

```python
# Async 模式伪代码
async def run_agent_loop(agent_id, prompt):
    # 每个 Agent 都在自己的独立世界里跑循环
    while not done:
        # Round 1
        # 这里的 await 只挂起当前这个 Agent，不影响别人
        response = await llm_server.generate(prompt) 
        
        # Agent 2 跑得快，瞬间执行完工具
        observation = await run_tool(response)
        
        # 更新自己的 Prompt
        prompt = update_history(prompt, response, observation)
        
        # 【关键点】：Agent 2 直接进入 Round 2 发起请求！
        # 此时 Agent 1 可能还在 Round 1 的 llm_server.generate 里没出来呢
        # 但没关系，vLLM 可以在处理 Agent 1 Round 1 的同时，
        # 接纳 Agent 2 Round 2 的请求
```

Async Rollout：
- 客户端层面：asyncio，“非阻塞地发射请求”。
    - 负责在 CPU 上并发处理业务逻辑（工具调用、Prompt 拼接），快速产生新的 Request。
- 服务端层面：Continuous Batching）- 配合 asyncio
    - `from vllm.engine.async_llm_engine import AsyncLLMEngine`
    - 负责在 GPU 上利用 Continuous Batching 技术，见缝插针地执行这些 Request，消灭了同步等待带来的算力空窗期。

### agent loop output

```python
class AgentLoopOutput(BaseModel):
    """Agent loop output."""

    prompt_ids: list[int]
    """Prompt token ids."""
    response_ids: list[int]
    """Response token ids including LLM generated token, tool response token."""
    response_mask: list[int]
    """Response mask, 1 for LLM generated token, 0 for tool response token."""
    response_logprobs: Optional[list[float]] = None
    """Log probabilities for the response tokens."""
    multi_modal_data: Optional[dict[str, Any]] = None
    """Multi-modal data for multi-modal tools."""
    reward_score: Optional[float] = None
    """Reward score for the trajectory."""
    num_turns: int = 0
    """Number of chat turns, including user, assistant, tool."""
    metrics: AgentLoopMetrics
    """Auxiliary performance metrics"""
    extra_fields: dict[str, Any] = {}
    """Extra fields for dynamic addition."""

```

### turn scores/rewards 

> retool.py

- AgentLoopWorkerBase:_run_agent_loop
    - `result = await self.reward_manager_worker.compute_score.remote(data)`
- AgentLoopWorkerBase:_postprocess
    - `rm_scores[torch.arange(response_mask.size(0)), response_length] = torch.tensor(scores, dtype=torch.float32)`

```python
def compute_score(data_source, solution_str, ground_truth, extra_info, **kwargs):  
    # 1. 首先计算答案正确性
    result = math_dapo.compute_score(solution_str, ground_truth, strict_box_verify=True)
    # 2. 使用 num_turns 调整 reward（鼓励工具调用）                                                                                                                       
    num_turns = extra_info["num_turns"]                                                                                                                                   
    if result["score"] < 0:
      tool_call_reward = (num_turns - 2) / 2 * 0.1  # 更多轮次 = 更高奖励
      result["score"] = min(-0.6, result["score"] + tool_call_reward)                                                                                  
    return result
```

$$ 
R =
\begin{cases}
1, & \text{if } S_{\text{base}} \ge 0 \quad (\text{correct}), \\
\min\left(-0.6, \, -1 + \frac{N-2}{2}\cdot 0.1\right), & \text{if } S_{\text{base}} < 0 \quad (\text{incorrect}).
\end{cases}
$$

- https://www.notion.so/verl-reTool-recipe-2398b5b7feba80a58156fa936f9f8de6
    - $S_{base}$ 为基础得分（由 math_dapo.compute_score 计算得出），其中：
        - 若答案正确，$S_{base} = 1$
        - 若答案错误，$S_{base} = -1$
    - $N$ 为交互轮数 num_turns
    - $R$ 为最终奖励值
- `naive.py`：Reward 放在 response 的最后一个有效 token 位置                                                                                          
    - `reward_tensor[i, valid_response_length - 1] = reward`
    - 这意味着：整个 trajectory 只在最后一个位置有非零 reward，前面都是 0。

### continuous batching

```
传统 batching：                                                                                                                                                    
Batch 1: [req1, req2, req3, req4] → 全部完成 → 输出                                                                                                               
Batch 2: [req5, req6, req7, req8] → 全部完成 → 输出                                                                                                               

Continuous batching：                                                                                                                                              
时间线:                                                                                                                                                            
t0: [req1, req2, req3, req4] 开始                                                                                                                                 
t1: req1 完成 → [req2, req3, req4, req5] ← req5 立即插入                                                                                                           
t2: req3 完成 → [req2, req4, req5, req6] ← req6 立即插入     
```

### sticky session (AsyncLLMServerManager)

```python
# agent_loop.py:60-88
class AsyncLLMServerManager:
    def __init__(self, config, server_handles, max_cache_size=10000):
        self.server_handles = server_handles  # Ray actor handles

        # 最小堆实现负载均衡
        self.weighted_serveres = [[0, (hash(server), server)] for server in server_handles]
        heapq.heapify(self.weighted_serveres)

        # LRU 缓存维护 request_id -> server 映射
        # 同一个 trajectory 的多轮对话路由到同一个 server       
        self.request_id_to_server = LRUCache(maxsize=max_cache_size)

    def _choose_server(self, request_id: str) -> ray.actor.ActorHandle:
        # 1. 如果已有映射, 返回同一个 server (sticky)
        if request_id in self.request_id_to_server:
            return self.request_id_to_server[request_id]

        # 2. 新请求, 选负载最低的 server
        server = self.weighted_serveres[0][1][1]
        self.weighted_serveres[0][0] += 1
        heapq.heapreplace(self.weighted_serveres, self.weighted_serveres[0])

        # 3. 记录映射
        self.request_id_to_server[request_id] = server
        return server
```

最小请求数负载均衡 (Least Requests Load Balancing)。
- 由于 self.weighted_serveres 是一个最小堆（Min-Heap），它的堆顶（索引 0）始终存放著当前负载（请求数）最小的那个服务器。

`self.request_id_to_server = LRUCache(maxsize=max_cache_size)`: # 默认 10000                                                                                      
- 当活跃 trajectory 数量 > 10000 时，最早的映射被淘汰
- 被淘汰的 trajectory 如果继续请求，会重新分配 server
- 但 SGLang 内部的 KV cache 也会基于 LRU 淘汰，所以影响不大   